# Lab #4 (Part 4) 과제 정답지: Optuna를 이용한 자전거 수요 예측 모델 최적화

In [13]:
# 필요한 라이브러리 임포트
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna

from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold

# ---------------------------------------------------
# 1. 데이터 로드 및 준비 (Data Loading and Preparation)
# ---------------------------------------------------
# 데이터셋 경로
path = '../../datasets/ml/bike-sharing/SeoulBikeData.csv'

# 데이터 로드 (원본 데이터 인코딩이 'latin1'으로 되어있음)
df = pd.read_csv(path, encoding='cp949')

# 특성(X)과 타겟(y) 분리
# 'Date', 'Functioning Day' 컬럼은 분석에서 제외
# 'Rented Bike Count'가 타겟 변수
X = df.drop(columns=['Date', 'Rented Bike Count', 'Functioning Day'])
y = df['Rented Bike Count']

# 범주형 변수를 원-핫 인코딩으로 변환
X = pd.get_dummies(X, columns=['Seasons', 'Holiday'], drop_first=True)

print("데이터 준비 완료!")
print("X shape:", X.shape)
print("y shape:", y.shape)

데이터 준비 완료!
X shape: (8760, 13)
y shape: (8760,)


### 2. Objective 함수 정의

In [14]:
# [문제 1] Objective 함수를 정의하세요. (정답)
def objective(trial):
    # --- 하이퍼파라미터 탐색 공간 정의 ---
    params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'n_estimators': trial.suggest_int('n_estimators', 100, 2000),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'random_state': 42,
        'n_jobs': -1
    }
    
    # XGBoost 모델 생성
    model = xgb.XGBRegressor(**params)
    
    # 3-Fold 교차 검증 설정
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    
    # 교차 검증 수행 (scoring='neg_root_mean_squared_error')
    scores = cross_val_score(model, X, y, cv=kf, scoring='neg_root_mean_squared_error')
    
    # 교차 검증 점수의 평균을 계산하고, RMSE를 반환 (scores는 음수 값이므로 양수로 변환)
    rmse = -np.mean(scores)
    
    return rmse

### 3. 최적화 실행 및 결과 확인

In [ ]:
# [문제 2] Optuna Study 객체를 생성하세요. (정답)
study = optuna.create_study(direction='minimize')

# [문제 3] 최적화를 100번의 trial로 실행하세요. (정답)
# n_trials를 50으로 줄여서 실행 시간을 단축할 수 있습니다.
study.optimize(objective, n_trials=100)

# [문제 4] 최적화 결과를 출력하세요. (정답)
print("--- 최적화 완료 ---")
print(f"최저 RMSE: {study.best_value:.4f}")
print("최적 하이퍼파라미터:", study.best_params)

### 4. 결과 분석 및 시각화

In [ ]:
# [문제 5] 최적화 과정을 보여주는 'Optimization History' 그래프를 그리세요. (정답)
from optuna.visualization import plot_optimization_history

fig1 = plot_optimization_history(study)
fig1.show()

In [ ]:
# [문제 6] 하이퍼파라미터들의 중요도를 보여주는 'Parameter Importances' 그래프를 그리세요. (정답)
from optuna.visualization import plot_param_importances

fig2 = plot_param_importances(study)
fig2.show()

### 🌟 [심화 문제] 결과 분석 및 해석 (정답 예시)

**답:** 

(실행 결과에 따라 순서는 달라질 수 있으나, 일반적으로 다음과 같은 파라미터들이 중요하게 나타납니다.)

1.  **`max_depth` (트리 최대 깊이)**: 모델의 복잡도를 직접적으로 제어하는 가장 중요한 파라미터 중 하나입니다. `max_depth`가 너무 얕으면 모델이 데이터의 복잡한 패턴을 충분히 학습하지 못하고(과소적합), 너무 깊으면 학습 데이터에만 과도하게 최적화되어(과적합) 일반화 성능이 떨어집니다. 이 파라미터가 최적의 균형점을 찾는 데 핵심적인 역할을 했기 때문에 중요도가 높게 나타났습니다.

2.  **`n_estimators` (트리 개수)**: Boosting 앙상블에서 트리의 개수는 모델의 성능과 직결됩니다. 트리가 많아질수록 모델은 점진적으로 오차를 줄여나가며 성능이 향상되는 경향이 있습니다. 하지만 일정 수준을 넘어서면 성능 향상은 미미해지고 계산 시간만 늘어납니다. 최적의 트리 개수를 찾는 것이 성능과 효율성 모두에 큰 영향을 미치므로 중요도가 높습니다.

3.  **`learning_rate` (학습률)**: 각 트리가 이전 트리의 오차를 얼마나 강하게 보정할지를 결정하는 파라미터입니다. 학습률이 낮으면 더 많은 트리가 필요하지만 보통 더 안정적이고 좋은 성능을 내는 경향이 있습니다. 학습률과 `n_estimators`는 서로 밀접한 관계(trade-off)가 있으며, 이 둘의 최적 조합을 찾는 것이 모델의 최종 성능을 결정하는 데 매우 중요하기 때문에 높은 중요도를 보입니다.